# 🧠 Semana 2 — Implementación de Redes Neuronales Básicas

**Curso:** CADI Deep Learning  
**Objetivo:** Implementar y validar una red neuronal básica (perceptrón) para entender los fundamentos del aprendizaje profundo: entradas, pesos, sesgo, puntaje z, función de activación y clasificación binaria.

---

## 📌 ¿Qué vamos a construir?

Un **perceptrón** es la unidad mínima de una red neuronal. Su funcionamiento se resume en tres pasos:

1. **Recibe entradas** → valores numéricos que representan los datos de entrada.
2. **Calcula el puntaje Z** → suma ponderada de entradas más un sesgo:
$$z = (x_1 \cdot w_1) + (x_2 \cdot w_2) + \cdots + b$$
3. **Aplica una función de activación** → decide si la salida es `0` o `1`.

En esta práctica implementaremos el perceptrón para simular la **compuerta lógica AND**, luego extenderemos el análisis añadiendo backpropagation y funciones de activación alternativas (Sigmoide y ReLU).

---
## 🔧 Parte 1 — Perceptrón básico (una neurona)

Comenzamos con la implementación más simple: **una sola neurona con función de activación escalón** (*step function* o Heaviside).

### ¿Por qué la función escalón?
Es la más intuitiva: si el puntaje `z` supera cierto umbral (en este caso `0`), la neurona "se activa" y devuelve `1`; de lo contrario devuelve `0`. Es perfecta para clasificación binaria pura.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─────────────────────────────────────────────
# FUNCIÓN: Perceptrón con función de activación escalón
# ─────────────────────────────────────────────
def perceptron(entradas, pesos, sesgo):
    """
    Simula una neurona artificial básica.

    Parámetros:
      entradas : lista de valores de entrada [x1, x2, ...]
      pesos    : lista de pesos asociados   [w1, w2, ...]
      sesgo    : valor del bias (b)

    Retorna:
      z      : puntaje z (suma ponderada + sesgo)
      salida : clasificación binaria (0 ó 1)
    """
    # Paso 1: suma ponderada de entradas
    z = sum(x * w for x, w in zip(entradas, pesos)) + sesgo
    # Paso 2: función de activación escalón (Heaviside)
    salida = 1 if z >= 0 else 0
    return z, salida


# ─────────────────────────────────────────────
# PARÁMETROS: Compuerta AND
# ─────────────────────────────────────────────
# Intuición: pesos iguales dan el mismo "peso de decisión" a cada entrada.
# El sesgo negativo actúa como umbral: solo cuando AMBAS entradas son 1
# la suma supera el umbral y la neurona se activa.
pesos = [0.6, 0.6]
sesgo = -1.0

# ─────────────────────────────────────────────
# CASOS DE PRUEBA: todas las combinaciones posibles de AND
# ─────────────────────────────────────────────
casos = [[0, 0], [0, 1], [1, 0], [1, 1]]
esperado = [0, 0, 0, 1]   # tabla de verdad de AND

resultados = []
for entradas, valor_esperado in zip(casos, esperado):
    z, y_hat = perceptron(entradas, pesos, sesgo)
    correcto = "✅" if y_hat == valor_esperado else "❌"
    resultados.append({
        "Entradas": str(entradas),
        "Puntaje Z": round(z, 2),
        "Salida (ŷ)": y_hat,
        "Esperado (y)": valor_esperado,
        "Correcto": correcto
    })

df = pd.DataFrame(resultados)
print("╔══════════════════════════════════════════════════════╗")
print("║        RESULTADOS — Compuerta AND (Perceptrón)       ║")
print("╚══════════════════════════════════════════════════════╝")
print(df.to_string(index=False))

---
## 📊 Parte 2 — Funciones de activación: Sigmoide y ReLU

La función escalón tiene una limitación: no es diferenciable, por lo que **no se puede usar directamente con backpropagation**. Las redes neuronales modernas usan funciones diferenciables:

| Función | Fórmula | Rango | Uso típico |
|---------|---------|-------|------------|
| **Sigmoide** | $\sigma(z) = \frac{1}{1 + e^{-z}}$ | (0, 1) | Capas de salida binaria |
| **ReLU** | $f(z) = \max(0, z)$ | $[0, +\infty)$ | Capas ocultas |

Veamos cómo se comportan visualmente y cómo cambia la salida del perceptrón al usarlas.

In [ ]:
# ─────────────────────────────────────────────
# FUNCIONES DE ACTIVACIÓN
# ─────────────────────────────────────────────

def sigmoide(z):
    """Convierte z en una probabilidad entre 0 y 1."""
    return 1 / (1 + np.exp(-z))

def relu(z):
    """Pasa valores positivos tal cual; convierte negativos en 0."""
    return max(0, z)

def escalon(z):
    """Retorna 1 si z >= 0, de lo contrario 0."""
    return 1 if z >= 0 else 0


# ─────────────────────────────────────────────
# VISUALIZACIÓN: comparación de las tres funciones
# ─────────────────────────────────────────────
z_vals = np.linspace(-5, 5, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Comparación de Funciones de Activación", fontsize=14, fontweight='bold')

configuraciones = [
    ("Escalón (Heaviside)",  [escalon(z) for z in z_vals],  "#2196F3"),
    ("Sigmoide",              sigmoide(z_vals),               "#4CAF50"),
    ("ReLU",                  [relu(z) for z in z_vals],      "#FF5722"),
]

for ax, (titulo, valores, color) in zip(axes, configuraciones):
    ax.plot(z_vals, valores, color=color, linewidth=2.5)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_title(titulo, fontsize=12)
    ax.set_xlabel("Puntaje z")
    ax.set_ylabel("Salida")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────
# TABLA COMPARATIVA: mismo perceptrón, distintas activaciones
# ─────────────────────────────────────────────
print("\n╔══════════════════════════════════════════════════════════════════╗")
print("║      Salida del perceptrón AND con distintas activaciones        ║")
print("╚══════════════════════════════════════════════════════════════════╝")

comp = []
for entradas in casos:
    z, _ = perceptron(entradas, pesos, sesgo)
    comp.append({
        "Entradas": str(entradas),
        "Puntaje Z": round(z, 2),
        "Escalón": escalon(z),
        "Sigmoide": round(float(sigmoide(z)), 4),
        "ReLU": round(relu(z), 4)
    })

print(pd.DataFrame(comp).to_string(index=False))

---
## 🔁 Parte 3 — Backpropagation: aprendiendo de los errores

Hasta ahora los pesos fueron fijados manualmente. En la realidad, la red **aprende sola** ajustando los pesos mediante **backpropagation** (retropropagación del error).

### Idea central:
1. La red hace una predicción `ŷ`.
2. Se calcula el **error** = diferencia entre `y` (real) e `ŷ` (predicho).
3. Los pesos se ajustan en dirección contraria al error, escalado por la **tasa de aprendizaje** (α):

$$w_i \leftarrow w_i + \alpha \cdot error \cdot x_i$$
$$b \leftarrow b + \alpha \cdot error$$

Repetimos este proceso varias **épocas** hasta que los pesos convergen.

In [ ]:
# ─────────────────────────────────────────────
# PERCEPTRÓN CON BACKPROPAGATION
# ─────────────────────────────────────────────
# Partimos de pesos aleatorios: la red NO conoce la solución de antemano.
# Ajustamos los pesos iteración a iteración usando la regla delta.

def entrenar_perceptron(X, y, tasa_aprendizaje=0.1, epocas=20, semilla=42):
    """
    Entrena un perceptrón usando la regla de actualización de pesos.

    Parámetros:
      X               : lista de listas con las entradas
      y               : lista de etiquetas correctas (0 ó 1)
      tasa_aprendizaje: tamaño del paso en cada actualización (alpha)
      epocas          : número de pasadas completas por el dataset
      semilla         : para reproducibilidad

    Retorna:
      pesos_final, sesgo_final, historial_error
    """
    np.random.seed(semilla)
    pesos = np.random.uniform(-0.5, 0.5, size=len(X[0]))   # pesos iniciales aleatorios
    sesgo = np.random.uniform(-0.5, 0.5)                   # sesgo inicial aleatorio

    historial_error = []   # guardamos el error total por época para graficar

    for epoca in range(epocas):
        error_total = 0

        for entradas, etiqueta in zip(X, y):
            # Predicción actual con los pesos en curso
            z = np.dot(entradas, pesos) + sesgo
            y_hat = 1 if z >= 0 else 0

            # Cálculo del error puntual
            error = etiqueta - y_hat
            error_total += abs(error)

            # Actualización de pesos y sesgo (regla delta)
            pesos += tasa_aprendizaje * error * np.array(entradas)
            sesgo += tasa_aprendizaje * error

        historial_error.append(error_total)

    return pesos, sesgo, historial_error


# ─────────────────────────────────────────────
# ENTRENAMIENTO
# ─────────────────────────────────────────────
X_train = [[0,0],[0,1],[1,0],[1,1]]
y_train = [0, 0, 0, 1]             # tabla de verdad AND

pesos_aprendidos, sesgo_aprendido, historial = entrenar_perceptron(
    X_train, y_train, tasa_aprendizaje=0.1, epocas=20
)

print(f"Pesos aprendidos : {np.round(pesos_aprendidos, 4)}")
print(f"Sesgo aprendido  : {round(sesgo_aprendido, 4)}")


# ─────────────────────────────────────────────
# GRÁFICA: evolución del error de entrenamiento
# ─────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(historial)+1), historial, marker='o', color='#3F51B5', linewidth=2)
plt.title("Evolución del Error durante el Entrenamiento", fontsize=13, fontweight='bold')
plt.xlabel("Época")
plt.ylabel("Error total (|y - ŷ|)")
plt.xticks(range(1, len(historial)+1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────
# VALIDACIÓN con los pesos aprendidos
# ─────────────────────────────────────────────
print("\n╔══════════════════════════════════════════════════════╗")
print("║    VALIDACIÓN — Pesos aprendidos por backpropagation ║")
print("╚══════════════════════════════════════════════════════╝")

val = []
for entradas, etiqueta in zip(X_train, y_train):
    z, y_hat = perceptron(entradas, list(pesos_aprendidos), sesgo_aprendido)
    correcto = "✅" if y_hat == etiqueta else "❌"
    val.append({
        "Entradas": str(entradas),
        "Puntaje Z": round(z, 4),
        "Salida (ŷ)": y_hat,
        "Esperado (y)": etiqueta,
        "Correcto": correcto
    })

print(pd.DataFrame(val).to_string(index=False))

---
## 🏗️ Parte 4 — Red multicapa (una capa oculta)

Un perceptrón simple **no puede** resolver problemas que no son linealmente separables (como XOR). Para eso necesitamos una **red multicapa** con al menos una **capa oculta**.

### Arquitectura que vamos a implementar:
```
Entrada [x1, x2]  →  Capa oculta (2 neuronas, ReLU)  →  Capa de salida (1 neurona, Sigmoide)  →  ŷ
```

- **Capa oculta:** extrae patrones intermedios de los datos.
- **Sigmoide en la salida:** convierte el resultado en una probabilidad (0–1), ideal para clasificación binaria.

In [ ]:
# ─────────────────────────────────────────────
# RED MULTICAPA: forward pass + backpropagation
# ─────────────────────────────────────────────
# Implementamos la red desde cero para entender cada operación.
# No usamos frameworks (TensorFlow/PyTorch) para que todo sea transparente.

class RedNeuronalBinaria:
    """
    Red neuronal de una capa oculta para clasificación binaria.

    Arquitectura: entrada → capa oculta (ReLU) → capa de salida (Sigmoide)
    Entrenamiento: descenso de gradiente + backpropagation manual.
    """

    def __init__(self, n_entradas, n_ocultas, tasa_aprendizaje=0.1, semilla=42):
        np.random.seed(semilla)
        # Pesos y sesgos de la capa oculta
        self.W1 = np.random.randn(n_entradas, n_ocultas) * 0.5
        self.b1 = np.zeros(n_ocultas)
        # Pesos y sesgos de la capa de salida
        self.W2 = np.random.randn(n_ocultas, 1) * 0.5
        self.b2 = np.zeros(1)
        self.lr = tasa_aprendizaje

    def _relu(self, z):
        return np.maximum(0, z)

    def _relu_deriv(self, z):
        """Derivada de ReLU: 1 donde z > 0, 0 en el resto."""
        return (z > 0).astype(float)

    def _sigmoide(self, z):
        return 1 / (1 + np.exp(-z))

    def forward(self, X):
        """Propagación hacia adelante: calcula la predicción."""
        # Capa oculta
        self.z1 = X @ self.W1 + self.b1    # suma ponderada
        self.a1 = self._relu(self.z1)       # activación ReLU
        # Capa de salida
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self._sigmoide(self.z2)   # activación Sigmoide → probabilidad
        return self.a2

    def backward(self, X, y):
        """
        Backpropagation: calcula gradientes y actualiza pesos.
        Usamos Binary Cross-Entropy como función de pérdida.
        """
        m = len(X)   # número de ejemplos

        # ── Gradientes capa de salida ──
        dL_da2 = self.a2 - y.reshape(-1, 1)          # derivada de la pérdida
        dW2    = (self.a1.T @ dL_da2) / m
        db2    = np.mean(dL_da2, axis=0)

        # ── Gradientes capa oculta (retropropagación) ──
        dL_da1 = (dL_da2 @ self.W2.T) * self._relu_deriv(self.z1)
        dW1    = (X.T @ dL_da1) / m
        db1    = np.mean(dL_da1, axis=0)

        # ── Actualización de pesos (descenso de gradiente) ──
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def entrenar(self, X, y, epocas=500):
        """Ciclo de entrenamiento completo."""
        historial = []
        for _ in range(epocas):
            y_hat = self.forward(X)
            # Binary Cross-Entropy
            eps = 1e-8   # evitar log(0)
            perdida = -np.mean(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))
            historial.append(perdida)
            self.backward(X, y)
        return historial

    def predecir(self, X, umbral=0.5):
        probs = self.forward(X)
        return (probs >= umbral).astype(int).flatten()


# ─────────────────────────────────────────────
# DATASET: clasificación binaria (compuerta AND)
# ─────────────────────────────────────────────
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([0, 0, 0, 1], dtype=float)

# ─────────────────────────────────────────────
# ENTRENAMIENTO
# ─────────────────────────────────────────────
red = RedNeuronalBinaria(n_entradas=2, n_ocultas=4, tasa_aprendizaje=0.3)
historial_perdida = red.entrenar(X, y, epocas=500)


# ─────────────────────────────────────────────
# GRÁFICA: curva de pérdida
# ─────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(historial_perdida, color='#E91E63', linewidth=2)
plt.title("Curva de Pérdida — Red Multicapa (Binary Cross-Entropy)", fontsize=13, fontweight='bold')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────
# RESULTADOS FINALES
# ─────────────────────────────────────────────
predicciones = red.predecir(X)
probabilidades = red.forward(X).flatten()

print("╔═══════════════════════════════════════════════════════════════╗")
print("║         RESULTADOS — Red Multicapa (AND dataset)              ║")
print("╚═══════════════════════════════════════════════════════════════╝")

res_multi = []
for entradas, prob, y_hat, etiqueta in zip(X.tolist(), probabilidades, predicciones, y.tolist()):
    correcto = "✅" if int(y_hat) == int(etiqueta) else "❌"
    res_multi.append({
        "Entradas": str([int(e) for e in entradas]),
        "Probabilidad (σ)": round(prob, 4),
        "Predicción (ŷ)": int(y_hat),
        "Esperado (y)": int(etiqueta),
        "Correcto": correcto
    })

print(pd.DataFrame(res_multi).to_string(index=False))

precision = sum(1 for p, e in zip(predicciones, y) if p == e) / len(y) * 100
print(f"\n🎯 Precisión final: {precision:.0f}%")

---
## 📝 Parte 5 — Análisis y Conclusiones

### 5.1 Efecto de los pesos y el sesgo en la decisión

En el **perceptrón manual** con pesos `[0.6, 0.6]` y sesgo `-1.0`:

| Caso | z = 0.6·x1 + 0.6·x2 − 1 | Decisión |
|------|--------------------------|----------|
| [0,0] | z = −1.0 | 0 (no activa) |
| [0,1] | z = −0.4 | 0 (no activa) |
| [1,0] | z = −0.4 | 0 (no activa) |
| [1,1] | z = +0.2 | 1 (activa ✅) |

El **sesgo** actúa como un umbral desplazado: al ser negativo (`-1.0`), exige que la suma de entradas supere ese valor antes de activarse. Sin él, la neurona se activaría incluso con entradas `[0,0]`.

Los **pesos iguales** (`0.6`) reflejan que ambas entradas tienen el mismo valor de decisión, lo que es coherente con la lógica AND.

### 5.2 Comparación de funciones de activación

- **Escalón:** útil solo para clasificación binaria perfecta; no es diferenciable → incompatible con backpropagation.
- **Sigmoide:** produce probabilidades suaves; diferenciable → compatible con backpropagation. Puede sufrir *vanishing gradient* en redes profundas.
- **ReLU:** soluciona el problema del gradiente desvanecido en capas ocultas; es simple y eficiente computacionalmente.

### 5.3 Impacto del backpropagation

Con **backpropagation**, la red ajusta los pesos automáticamente partiendo de valores aleatorios. La gráfica de error muestra cómo el modelo converge: en las primeras épocas el error es alto y cae rápidamente conforme los pesos se aproximan a los valores óptimos.

### 5.4 Ventaja de la red multicapa

La red multicapa con capa oculta **no solo clasifica AND**, sino que tiene capacidad de aprender patrones más complejos (como XOR) que un perceptrón simple no puede resolver. La capa oculta transforma el espacio de entrada en una representación en la que el problema se vuelve linealmente separable.